# Is it a cat or a dog?

Upload a photo and click **Classify**.

In [ ]:
# #hide
# import sys

# !pip install -Uqq fastai
# # !pip install jupyterlab
# !pip install -U jupyterlab notebook ipywidgets
# !pip install voila
# !jupyter server extension enable --sys-prefix voila 
# !{sys.executable} -m pip install -U voila
# # !jupyter serverextension enable --sys-prefix voila
# # !jupyter server extension enable voila --sys-prefix
# # !jupyter server extension enable voila --sys-prefix
# !{sys.executable} -m jupyter server extension enable voila.server_extension --sys-prefix

In [ ]:
# !pip list
# !pip show fastai
# !pip show ipywidgets

In [ ]:
from io import BytesIO
import plum._resolver
# Models exported on Python 3.12 pickle plum Resolver state via __dict__.
# plum>=2.9 uses __slots__, which breaks unpickling on Python 3.13+.
class _ResolverCompat(plum._resolver.Resolver):
    def __setstate__(self, state):
        if isinstance(state, dict):
            for key, value in state.items():
                setattr(self, key, value)
plum._resolver.Resolver = _ResolverCompat

In [ ]:
from pathlib import Path

import torch
from fastai.vision.all import PILImage, load_learner
from IPython.display import display
from ipywidgets import VBox, widgets

In [ ]:
import warnings
from pathlib import Path

model_path = Path('models/01-cat-or-dog-model.pkl')
if not model_path.exists():
    # Binder/Voilà cwd is usually the repo root; fall back to this file's directory.
    model_path = Path.cwd() / 'models' / '01-cat-or-dog-model.pkl'
if not model_path.exists():
    raise FileNotFoundError(f'Model not found: {model_path.resolve()}')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    learn_inf = load_learner(model_path)

# CPU inference is more stable than MPS for single-image predict in notebooks.
learn_inf.dls.cpu()
_ = learn_inf.model.cpu()

In [ ]:
# path = Path()
# learn_inf = load_learner('cat-or-dog-model.pkl')

In [ ]:
_CLASS_NAMES = {False: 'Dog', True: 'Cat'}


def safe_predict(learn, img):
    """Run inference without learn.predict() (crashes on Python 3.14)."""
    learn.model.eval()
    x = learn.dls.after_item(img)
    xb = learn.dls.after_batch(torch.stack([x]))
    with torch.inference_mode():
        probs = learn.model(xb).softmax(dim=-1)[0]
    pred_idx = int(probs.argmax())
    return learn.dls.vocab[pred_idx], pred_idx, probs


btn_upload = widgets.FileUpload()
btn_run = widgets.Button(description='Classify')
lbl_pred = widgets.Label(value='Upload an image, then click Classify.')
out_pl = widgets.Output()


def on_click_classify(_change):
    if not btn_upload.value:
        lbl_pred.value = 'Please upload an image first.'
        return

    upload = btn_upload.value[0]
    content = upload['content']
    if isinstance(content, memoryview):
        content = bytes(content)

    try:
        img = PILImage.create(BytesIO(content))
        out_pl.clear_output(wait=True)
        with out_pl:
            display(img.to_thumb(128, 128))

        pred, pred_idx, probs = safe_predict(learn_inf, img)
        label = _CLASS_NAMES.get(pred, str(pred))
        lbl_pred.value = f'Prediction: {label}; Probability: {float(probs[pred_idx]):.04f}'
    except Exception as exc:
        lbl_pred.value = f'Prediction failed: {exc}'


btn_run.on_click(on_click_classify)

display(VBox([
    widgets.Label('Select your pic!'),
    btn_upload,
    btn_run,
    out_pl,
    lbl_pred,
]))

In [ ]:
# UI is built in the cell above (widgets + display(VBox(...))).
# Re-run that cell if you need to reset the app.

In [ ]:
# print(lbl_pred)

In [ ]:
# !voila web-app.ipynb

In [ ]:
# One-time setup (run in a terminal, not in Voilà):
#   python -m pip install -U voila
#   python -m jupyter server extension enable voila.server_extension --sys-prefix
#   python -m voila 01_cat_or_dog.ipynb